# Concept Portfolio V2 Lab

## 처음 실행한다면

1. 아래 `MODE = "MOCK"`를 유지합니다.
2. Kernel Restart 후 **Run All**을 실행합니다.
3. Final Portfolio, Trace, Downstream Handoff를 확인합니다.
4. Partial Lock과 Heavy Lock 결과를 비교합니다.
5. 환경변수를 준비한 뒤에만 `LIVE`로 바꿉니다. LIVE는 Provider 비용이 발생할 수 있습니다.

이 Notebook은 입력·실행·표시만 담당합니다. 모든 business logic은 `app.concept_portfolio_v2` Python Core에 있습니다.

## 00–02. 목적과 환경 / import 확인

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from app.concept_portfolio_v2 import ConceptPortfolioEngine, ProviderGateway
from app.concept_portfolio_v2.diagnostics.notebook_view import *
print('AI root:', ROOT)
print('Python:', sys.version.split()[0])

AI root: C:\Users\seewo\Desktop\big_proj_01\new_3\ai
Python: 3.14.5


## 03–04. MODE / Provider / Legal 환경 확인
`LIVE`는 `AI_PROVIDER`, `AI_API_KEY`, `AI_MODEL`과 `MOLEG_API_KEY`가 필요합니다.

In [2]:
MODE = 'MOCK'  # MOCK | REPLAY | LIVE
RECORDINGS_DIR = ROOT / 'fixtures' / 'concept_portfolio_v2' / 'recordings'
print('LIVE Provider calls enabled' if MODE == 'LIVE' else f'{MODE} mode: external cost disabled')

MOCK mode: external cost disabled


## 05. 테스트 Seed 입력
필수: `ideaOverview`, `problem`, `targetUsers`. 선택값은 문자열 또는 `{value, decisionState, source}`로 입력합니다.

In [3]:
TEST_INPUT = {
    'fixtureName': 'food_minimal',
    'ideaOverview': '개인 맞춤형 식재료를 소량 제공하고 남은 재료 활용 레시피를 안내하는 서비스',
    'problem': '1~2인 가구가 식재료를 다 쓰지 못해 비용과 음식물 쓰레기가 발생한다',
    'targetUsers': '식재료 낭비를 줄이고 싶은 1~2인 가구',
    # 'price': {'value': '월 19,900원', 'decisionState': 'LOCKED', 'source': 'USER_INPUT'},
}
gateway = ProviderGateway(MODE, recordings_dir=RECORDINGS_DIR)
engine = ConceptPortfolioEngine(MODE, gateway=gateway)

## 06. 현재 Idea Brief Adapter 결과

In [4]:
seed = engine.seed_adapter.adapt(TEST_INPUT)
show_seed_input(seed)

,fieldKey,value,source,decisionState
0,ideaOverview,개인 맞춤형 식재료를 소량 제공하고 남은 재료 활용 레시피를 안내하는 서비스,USER_INPUT,LOCKED
1,problem,1~2인 가구가 식재료를 다 쓰지 못해 비용과 음식물 쓰레기가 발생한다,USER_INPUT,LOCKED
2,targetUsers,식재료 낭비를 줄이고 싶은 1~2인 가구,USER_INPUT,LOCKED
3,targetRegion,,MISSING,OPEN
4,knownCompetitors,,MISSING,OPEN
5,revenueModel,,MISSING,OPEN
6,price,,MISSING,OPEN
7,channels,,MISSING,OPEN
8,differentiators,,MISSING,OPEN
9,budgetConstraint,,MISSING,OPEN


## 07. Safety 결과
Safety가 차단하면 뒤 planning을 실행하면 안 됩니다.

In [5]:
safety = await engine.check_safety(seed)
safety

SafetyResult(decision='ALLOW', categories=[], restrictions=[], userFacingReason='안전한 사업 아이디어로 확인했습니다.')

## 08–09. Seed Analysis / Open Design Space

In [6]:
analysis = await engine.analyze_seed(seed)
show_seed_analysis(analysis), show_design_space(analysis)

(        구분                                   값
 0     탐색 폭                             EXPLORE
 1  다양성 수용량                                   5
 2       설명  선택 입력 LOCK 0개로 11개 설계 차원이 열려 있습니다.,
                  분류                  필드  \
 0         HARD_LOCK        ideaOverview   
 1         HARD_LOCK             problem   
 2         HARD_LOCK         targetUsers   
 3   SEMANTIC_ANCHOR        ideaOverview   
 4   SEMANTIC_ANCHOR             problem   
 5   SEMANTIC_ANCHOR         targetUsers   
 6              OPEN   solutionMechanism   
 7              OPEN       valueDelivery   
 8              OPEN      operatingModel   
 9              OPEN     supplyStructure   
 10             OPEN        partnerModel   
 11             OPEN     transactionFlow   
 12             OPEN         fulfillment   
 13             OPEN        platformRole   
 14             OPEN      commercialFlow   
 15             OPEN      dataDependency   
 16             OPEN  physicalDependency   
 
             

## 10–12. Portfolio Planning / Plan Diversity / Selection

In [7]:
plan_pool = await engine.plan_portfolio(seed, analysis, max_concepts=5)
plan_validation = await engine.validate_plans(plan_pool, analysis, max_concepts=5)
show_portfolio_plans(plan_validation.acceptedPlans), show_plan_diversity(plan_validation.diversity)

(  planId        제목        핵심 mechanics                  운영          파트너  \
 0     P1  주간 소분 구독       수요예측 기반 정기 소분       수요예측 기반 정기 소분      지역 소분센터   
 1     P2   동네 공동구매      이웃 수요를 묶는 공동주문      이웃 수요를 묶는 공동주문   지역 식자재점 제휴   
 2     P3    레시피 마켓  남은 재료 기반 레시피·재료 번들  남은 재료 기반 레시피·재료 번들  셰프·생산자 큐레이션   
 3     P4    냉장고 코치     보유 재료 인식과 보충 추천     보유 재료 인식과 보충 추천   리테일 데이터 제휴   
 4     P5  기업 복지 키트     직장 단위 맞춤 식재료 키트     직장 단위 맞춤 식재료 키트    기업·급식 파트너   
 
          거래        이행  
 0     직접 구독      정기배송  
 1  공동구매 수수료      거점수령  
 2     번들 판매   온디맨드 배송  
 3    프리미엄 앱     매장 픽업  
 4    B2B 계약  사무실 일괄배송  ,
      A   B        판정 겹침                                              실질 차이  \
 0   P1  P2  DISTINCT     mechanism, operation, partner, transaction, co...   
 1   P1  P3  DISTINCT     mechanism, operation, partner, transaction, co...   
 2   P2  P3  DISTINCT     mechanism, operation, partner, transaction, co...   
 3   P1  P4  DISTINCT     mechanism, operation, partner, transaction, co...

## 13–15. Full Candidate / Validation / Pairwise Distinctness

In [8]:
expanded = await engine.expand_plans(seed, plan_validation.acceptedPlans)
candidates, candidate_reports = await engine.validate_candidates(seed, plan_validation.acceptedPlans, expanded)
candidate_pairs = [engine.compare_candidates(candidates[i], candidates[j]) for i in range(len(candidates)) for j in range(i+1, len(candidates))]
show_candidates(candidates), show_candidate_validation(candidate_reports), compare_candidates(candidate_pairs)

(  candidateId lineageId parentCandidateId        이름             핵심 작동방식  \
 0          C1        L1              None  주간 소분 구독       수요예측 기반 정기 소분   
 1          C2        L2              None   동네 공동구매      이웃 수요를 묶는 공동주문   
 2          C3        L3              None    레시피 마켓  남은 재료 기반 레시피·재료 번들   
 3          C4        L4              None    냉장고 코치     보유 재료 인식과 보충 추천   
 4          C5        L5              None  기업 복지 키트     직장 단위 맞춤 식재료 키트   
 
          수익                  운영  
 0     직접 구독       수요예측 기반 정기 소분  
 1  공동구매 수수료      이웃 수요를 묶는 공동주문  
 2     번들 판매  남은 재료 기반 레시피·재료 번들  
 3    프리미엄 앱     보유 재료 인식과 보충 추천  
 4    B2B 계약     직장 단위 맞춤 식재료 키트  ,
   candidateId  schemaValid  hardLockPreserved  semanticAnchorPreserved  \
 0          C1         True               True                     True   
 1          C2         True               True                     True   
 2          C3         True               True                     True   
 3          C4         True    

## 16. Legal Structural Precheck
**Structural risk precheck — not final legal review**

In [9]:
prechecks = [engine.legal_precheck(item) for item in candidates]
show_legal_precheck(prechecks)

,candidateId,label,directSeller,intermediary,regulatedPhysicalActivity,personalDataDependency,qualificationDependency,riskHints
0,C1,Structural risk precheck — not final legal review,True,False,True,True,False,"[물리 활동, 개인정보]"
1,C2,Structural risk precheck — not final legal review,True,False,True,True,False,"[물리 활동, 개인정보]"
2,C3,Structural risk precheck — not final legal review,True,False,True,True,False,"[물리 활동, 개인정보]"
3,C4,Structural risk precheck — not final legal review,True,False,True,True,False,"[물리 활동, 개인정보]"
4,C5,Structural risk precheck — not final legal review,True,False,True,True,False,"[물리 활동, 개인정보]"


## 17–19. Full Legal / Redesign / Replan

In [10]:
legal = await engine.review_legal(seed, candidates)
portfolio, legal_all, required_inputs, redesigned, replanned = await engine.resolve_legal(seed, plan_validation.acceptedPlans, candidates, legal)
show_legal_result(legal_all), {'redesigned': redesigned, 'replanned': replanned, 'requiredInputs': required_inputs}

(  candidateId   route                  source                            요약  \
 0          C1  ACCEPT  MOCK_OFFICIAL_EVIDENCE  구조화된 MOCK 공식근거 계약상 수용 가능합니다.   
 1          C2  ACCEPT  MOCK_OFFICIAL_EVIDENCE  구조화된 MOCK 공식근거 계약상 수용 가능합니다.   
 2          C3  ACCEPT  MOCK_OFFICIAL_EVIDENCE  구조화된 MOCK 공식근거 계약상 수용 가능합니다.   
 3          C4  ACCEPT  MOCK_OFFICIAL_EVIDENCE  구조화된 MOCK 공식근거 계약상 수용 가능합니다.   
 4          C5  ACCEPT  MOCK_OFFICIAL_EVIDENCE  구조화된 MOCK 공식근거 계약상 수용 가능합니다.   
 
                  통제  
 0  표시·거래 조건을 명확히 고지  
 1  표시·거래 조건을 명확히 고지  
 2  표시·거래 조건을 명확히 고지  
 3  표시·거래 조건을 명확히 고지  
 4  표시·거래 조건을 명확히 고지  ,
 {'redesigned': 0, 'replanned': 0, 'requiredInputs': []})

## 20–21. Final Portfolio / Concept 선택

In [11]:
show_candidates(portfolio)
selected_concept = portfolio[0] if portfolio else None
selected_concept

CandidateEnvelope(candidateId='C1', planId='P1', lineageId='L1', parentCandidateId=None, redesignRound=0, candidate=ConceptCandidateResult(conceptName='주간 소분 구독', conceptDefinition='식재료 낭비를 줄이고 싶은 1~2인 가구를 위한 주간 소분 구독', introduction='사업 작동방식 1의 독립 대안', coreValue='1~2인 가구가 식재료를 다 쓰지 못해 비용과 음식물 쓰레기가 발생한다을 줄이는 수요예측 기반 정기 소분', targetUsers='식재료 낭비를 줄이고 싶은 1~2인 가구', industryCategory='푸드테크', researchScope='대한민국 식재료 구매·소비', targetRegion='대한민국', revenueModel='직접 구독', price='실험 후 확정', channels='주간 소분 구독 전용 모바일·현장 접점', differentiators='수요예측 기반 정기 소분, 지역 소분센터, 직접 구독, 정기배송', preMarketSomShareHypothesis=PreMarketSomShareHypothesis(targetSharePercent=2.0, horizonYears=3, rationale='시장 분석 전 검증할 가설', assumptions=['초기 가설']), preMarketSomHypothesis=PreMarketSomHypothesis(amount=100000000.0, currency='KRW', period='연간', calculationBasis='시장 분석 전 임시 산식', assumptions=['초기 가설'], confidence='LOW'), problemScenario='1~2인 가구가 식재료를 다 쓰지 못해 비용과 음식물 쓰레기가 발생한다', solutionMechanism='수요예측 기반 정기 소분', featureSet=['수요예측 

## 22. 7개 Hypothesis 확인 후 Downstream Handoff
Production 의미와 동일하게 7개 가설을 모두 확정한 뒤 payload를 만듭니다.

In [12]:
hypotheses = engine.build_or_load_current_hypothesis_contract(selected_concept) if selected_concept else []
show_hypotheses(hypotheses)

,hypothesisType,proposedValue,finalValue,source,decisionStatus,proposalVersion,locked,legalImpact,legalReviewStatus
0,TARGET_REGION,대한민국,None,CONCEPT_GENERATED,PROPOSED,1,False,NONE,NOT_REQUIRED
1,REVENUE_MODEL,직접 구독,None,CONCEPT_GENERATED,PROPOSED,1,False,NONE,NOT_REQUIRED
2,PRICE,실험 후 확정,None,CONCEPT_GENERATED,PROPOSED,1,False,NONE,NOT_REQUIRED
3,CHANNELS,주간 소분 구독 전용 모바일·현장 접점,None,CONCEPT_GENERATED,PROPOSED,1,False,NONE,NOT_REQUIRED
4,DIFFERENTIATORS,"수요예측 기반 정기 소분, 지역 소분센터, 직접 구독, 정기배송",None,CONCEPT_GENERATED,PROPOSED,1,False,NONE,NOT_REQUIRED
5,PRE_MARKET_SOM_SHARE,"{'targetSharePercent': 2.0, 'horizonYears': 3,...",None,AI_HYPOTHESIS,PROPOSED,1,False,NONE,NOT_REQUIRED
6,PRE_MARKET_SOM,"{'amount': 100000000.0, 'currency': 'KRW', 'pe...",None,AI_HYPOTHESIS,PROPOSED,1,False,NONE,NOT_REQUIRED


In [13]:
HYPOTHESIS_EDITS = {}  # 예: {'PRICE': '월 17,900원'}
confirmed_hypotheses = engine.confirm_hypotheses(hypotheses, HYPOTHESIS_EDITS)
handoff = engine.build_downstream_handoff(seed, selected_concept, confirmed_hypotheses, legal_all) if selected_concept else None
show_downstream_handoff(handoff) if handoff else '선택 가능한 Concept 없음'

{'호환성': 'PASS',
 '필드 매핑':                        v2Field                             downstreamField  \
 0        candidate.conceptName        selectedConcept.identity.conceptName   
 1  candidate.solutionMechanism  selectedConcept.solution.solutionMechanism   
 2                hypotheses[*]                             finalHypotheses   
 3                        legal                                 legalResult   
 
               source  transformed  required  
 0  CONCEPT_GENERATED        False      True  
 1  CONCEPT_GENERATED        False      True  
 2     USER_CONFIRMED         True      True  
 3  OFFICIAL_EVIDENCE         True      True  ,
 'Market payload': {'contract': 'market-analysis-seed-snapshot-v1',
  'schemaVersion': '2.0',
  'snapshotId': 'lab-market-seed',
  'projectId': 0,
  'selectionId': 0,
  'conceptId': 'C1',
  'createdAt': '2026-08-09T14:53:16.970644+00:00',
  'sourceSnapshotHash': 'sha256:06b4c2388558c56e83570fa6739ba40772b2855e62625dd24ce7ff1fa673fb98',
  'o

## 23–25. 전체 Trace / Provider Usage / Raw JSON

In [14]:
show_trace(engine.trace), show_provider_usage(engine.gateway.usage)

(    순서                                시각                 stage  \
 0    1  2026-08-09T14:53:16.800137+00:00       SAFETY_CHECKING   
 1    2  2026-08-09T14:53:16.800216+00:00       SAFETY_CHECKING   
 2    3  2026-08-09T14:53:16.809077+00:00        SEED_ANALYZING   
 3    4  2026-08-09T14:53:16.809172+00:00        SEED_ANALYZING   
 4    5  2026-08-09T14:53:16.840693+00:00              PLANNING   
 5    6  2026-08-09T14:53:16.841185+00:00              PLANNING   
 6    7  2026-08-09T14:53:16.841291+00:00       PLAN_VALIDATING   
 7    8  2026-08-09T14:53:16.842990+00:00       PLAN_VALIDATING   
 8    9  2026-08-09T14:53:16.861783+00:00             EXPANDING   
 9   10  2026-08-09T14:53:16.862146+00:00             EXPANDING   
 10  11  2026-08-09T14:53:16.862430+00:00             EXPANDING   
 11  12  2026-08-09T14:53:16.862629+00:00             EXPANDING   
 12  13  2026-08-09T14:53:16.862784+00:00             EXPANDING   
 13  14  2026-08-09T14:53:16.862924+00:00             EXPANDIN

In [15]:
print(show_raw_json(handoff)[:5000] if handoff else '{}')

{
  "compatibility": "PASS",
  "marketAnalysisSeedSnapshot": {
    "contract": "market-analysis-seed-snapshot-v1",
    "schemaVersion": "2.0",
    "snapshotId": "lab-market-seed",
    "projectId": 0,
    "selectionId": 0,
    "conceptId": "C1",
    "createdAt": "2026-08-09T14:53:16.970644+00:00",
    "sourceSnapshotHash": "sha256:06b4c2388558c56e83570fa6739ba40772b2855e62625dd24ce7ff1fa673fb98",
    "originalSeed": {
      "ideaOverview": "개인 맞춤형 식재료를 소량 제공하고 남은 재료 활용 레시피를 안내하는 서비스",
      "fields": {
        "ideaOverview": {
          "value": "개인 맞춤형 식재료를 소량 제공하고 남은 재료 활용 레시피를 안내하는 서비스",
          "source": "USER_INPUT",
          "decisionState": "LOCKED"
        },
        "problem": {
          "value": "1~2인 가구가 식재료를 다 쓰지 못해 비용과 음식물 쓰레기가 발생한다",
          "source": "USER_INPUT",
          "decisionState": "LOCKED"
        },
        "targetUsers": {
          "value": "식재료 낭비를 줄이고 싶은 1~2인 가구",
          "source": "USER_INPUT",
          "decisionState": "LOCKED"
        }
      }

## 26. One-click Full Run

In [16]:
result = await ConceptPortfolioEngine(MODE, gateway=ProviderGateway(MODE, recordings_dir=RECORDINGS_DIR)).run_full(TEST_INPUT)
show_run_summary(result)

,safety,requestedMaximum,planned,planDuplicatesRemoved,candidatesExpanded,legalAccepted,legalRedesigned,replanned,finalPortfolio,portfolioStatus,selectedConcept,downstreamHandoff,providerCalls,totalDurationMs
0,PASS,5,7,0,5,5,0,0,5,READY_FULL,주간 소분 구독,PASS,11,6


## 27–28. Partial Lock / Heavy Lock Scenario

In [17]:
def load_fixture(name):
    return json.loads((ROOT / 'fixtures' / 'concept_portfolio_v2' / f'{name}.json').read_text(encoding='utf-8'))
partial_result = await ConceptPortfolioEngine('MOCK').run_full(load_fixture('food_partial_lock'))
heavy_result = await ConceptPortfolioEngine('MOCK').run_full(load_fixture('food_heavy_lock'))
show_run_summary(partial_result), show_run_summary(heavy_result)

(  safety  requestedMaximum  planned  planDuplicatesRemoved  \
 0   PASS                 5        5                      0   
 
    candidatesExpanded  legalAccepted  legalRedesigned  replanned  \
 0                   3              3                0          0   
 
    finalPortfolio portfolioStatus selectedConcept downstreamHandoff  \
 0               3   READY_LIMITED        주간 소분 구독              PASS   
 
    providerCalls  totalDurationMs  
 0              7                3  ,
   safety  requestedMaximum  planned  planDuplicatesRemoved  \
 0   PASS                 5        4                      0   
 
    candidatesExpanded  legalAccepted  legalRedesigned  replanned  \
 0                   2              2                0          0   
 
    finalPortfolio portfolioStatus selectedConcept downstreamHandoff  \
 0               2   READY_LIMITED        주간 소분 구독              PASS   
 
    providerCalls  totalDurationMs  
 0              5                3  )

## 29. Replay Run
`MODE='REPLAY'`일 때 같은 canonical request hash의 LIVE 기록만 사용합니다. 기록이 없으면 `REPLAY_MISS`이며 MOCK으로 대체하지 않습니다.

In [18]:
if MODE == 'REPLAY':
    replay_result = await ConceptPortfolioEngine('REPLAY', gateway=ProviderGateway('REPLAY', recordings_dir=RECORDINGS_DIR)).run_full(TEST_INPUT)
    display(show_run_summary(replay_result))
else:
    print('REPLAY cell skipped: MODE is', MODE)

REPLAY cell skipped: MODE is MOCK


## 30. 결과 해석 방법
- `READY_FULL`: 요청 최대치(기본 5)의 유효 대안이 준비됨
- `READY_LIMITED`: 억지로 채우지 않고 1~4개의 유효 대안만 반환
- `NEEDS_INPUT`: LOCK/법률/입력 충돌을 사용자가 결정해야 함
- `FAILED`: Provider, schema, replay 등 시스템 실패

## 31. 알려진 제한사항
이 Lab은 production DB에 쓰지 않으며 route/UI를 변경하지 않습니다. LIVE Legal은 MOLEG 및 AI Provider 환경이 필요하고 실제 법률 자문이 아닙니다.